# Fusion Detection Cache — AMI Group 1

Generates `detections_fusion.json` for the 5 FRED fusion sequences using:
- `fusion_rgb.pt` — RGB component model (YOLOv8n)
- `fusion_event.pt` — Event component model (YOLOv8n, run2)

**Input dataset:** `ami-fusion-g1`  
Contains: `84.zip`, `85.zip`, `124.zip`, `127.zip`, `201.zip` + weights

**Runtime:** GPU T4 recommended (~5 min total)

**Output:** `detections_fusion.json` × 5 sequences in `/kaggle/working/`

In [ ]:
import re, json, zipfile, shutil
from pathlib import Path
import numpy as np

DATASET  = Path('/kaggle/input/ami-fusion-g1')
WORK     = Path('/kaggle/working')
SEQUENCES = [84, 85, 124, 127, 201]

print('Dataset contents:', sorted(f.name for f in DATASET.iterdir()))

In [ ]:
from ultralytics import YOLO

rgb_model   = YOLO(str(DATASET / 'fusion_rgb.pt'))
event_model = YOLO(str(DATASET / 'fusion_event.pt'))
print('Models loaded.')
print('  RGB   nc:', rgb_model.model.nc,   'names:', rgb_model.names)
print('  Event nc:', event_model.model.nc, 'names:', event_model.names)

In [ ]:
# ── Fusion layer (mirrors fusion_layer.py) ────────────────────────────────────
EVENT_THRESHOLD = 0.5
RGB_THRESHOLD   = 0.8
IOU_THRESHOLD   = 0.5

def _iou(a, b):
    ix1, iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2, iy2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0: return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

def fuse_detections(event_dets, rgb_dets):
    final = []
    for ev in event_dets:
        best_iou, best_rgb = 0, None
        for rgb in rgb_dets:
            iou_val = _iou(ev['box'], rgb['box'])
            if iou_val > best_iou:
                best_iou, best_rgb = iou_val, rgb
        if best_rgb and best_iou > IOU_THRESHOLD:
            final.append({'box': ev['box'],
                          'confidence': max(ev['confidence'], best_rgb['confidence']),
                          'source': 'fusion'})
        elif ev['confidence'] >= EVENT_THRESHOLD:
            final.append({'box': ev['box'], 'confidence': ev['confidence'], 'source': 'event'})
    for rgb in rgb_dets:
        if rgb['confidence'] >= RGB_THRESHOLD:
            final.append({'box': rgb['box'], 'confidence': rgb['confidence'], 'source': 'rgb'})
    return final

def _boxes(result):
    return [{'box': b.xyxy[0].tolist(), 'confidence': float(b.conf[0])}
            for b in result.boxes]

def _numeric_key(p):
    m = re.search(r'_(\d+)\.png$', p.name)
    return int(m.group(1)) if m else 0

In [ ]:
BATCH = 32
TMP   = WORK / '_tmp_extract'

for seq_n in SEQUENCES:
    zip_path = DATASET / f'{seq_n}.zip'
    seq_id   = f'sequence_{seq_n}'
    tmp_seq  = TMP / str(seq_n)
    out_path = WORK / seq_id / 'detections_fusion.json'

    print(f'\n=== {seq_id} ===')

    # ── Extract zip ─────────────────────────────────────────────────────────
    if tmp_seq.exists():
        shutil.rmtree(tmp_seq)
    tmp_seq.mkdir(parents=True)
    print(f'  Extracting {zip_path.name} ...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmp_seq)

    seq_dir   = tmp_seq / str(seq_n)
    rgb_dir   = seq_dir / 'RGB'
    event_dir = seq_dir / 'Event' / 'Frames'

    rgb_files   = sorted(rgb_dir.glob('*.jpg'))   if rgb_dir.exists()   else []
    event_files = sorted(event_dir.glob('*.png'), key=_numeric_key) if event_dir.exists() else []

    n_frames = min(len(rgb_files), len(event_files))
    print(f'  RGB={len(rgb_files)}  Event={len(event_files)}  paired={n_frames}')
    if n_frames == 0:
        print('  WARNING: no paired frames — skipping')
        shutil.rmtree(tmp_seq)
        continue

    # ── Run inference ────────────────────────────────────────────────────────
    detections = []
    for i in range(0, n_frames, BATCH):
        b_rgb = [str(f) for f in rgb_files[i:i+BATCH]]
        b_evt = [str(f) for f in event_files[i:i+BATCH]]

        rgb_results = rgb_model(b_rgb,   verbose=False, conf=0.01)
        evt_results = event_model(b_evt, verbose=False, conf=0.01)

        for j, (rgb_r, evt_r) in enumerate(zip(rgb_results, evt_results)):
            frame_idx = i + j
            fused = fuse_detections(_boxes(evt_r), _boxes(rgb_r))
            for det in fused:
                x1, y1, x2, y2 = det['box']
                detections.append({
                    'frame':      frame_idx,
                    'bbox':       [x1, y1, x2 - x1, y2 - y1],
                    'confidence': det['confidence'],
                    'class':      'drone',
                    'source':     det['source'],
                })

        if (i // BATCH) % 10 == 0 or i + BATCH >= n_frames:
            print(f'  {min(i+BATCH, n_frames)}/{n_frames} frames  '
                  f'({len(detections)} detections so far)')

    # ── Save JSON ────────────────────────────────────────────────────────────
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps({
        'sequence_id': seq_id,
        'model':       'fusion',
        'cached':      True,
        'detections':  detections,
    }, indent=2))
    print(f'  → {out_path.relative_to(WORK)}  ({len(detections)} detections)')

    shutil.rmtree(tmp_seq)

print('\n=== All sequences done ===')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
for seq_n in SEQUENCES:
    p = WORK / f'sequence_{seq_n}' / 'detections_fusion.json'
    if p.exists():
        d = json.loads(p.read_text())
        dets   = d['detections']
        by_src = {}
        for det in dets:
            src = det.get('source', '?')
            by_src[src] = by_src.get(src, 0) + 1
        print(f'sequence_{seq_n}: {len(dets):5d} total  {by_src}')
    else:
        print(f'sequence_{seq_n}: MISSING')